In [0]:
%pip install pendulum

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import pendulum

In [0]:
spark.sql("create catalog if not exists proyecto_final_prueba")

DataFrame[]

In [0]:
spark.sql("use catalog proyecto_final_prueba")


DataFrame[]

In [0]:
catalog = spark.sql("select current_catalog()").first()[0]
schema = "raw"
volume = "weather"

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

DataFrame[]

In [0]:
spark.sql(f"drop volume if exists {catalog}.{schema}.{volume}")

DataFrame[]

In [0]:
spark.sql(f"create volume if not exists {catalog}.{schema}.{volume}")

DataFrame[]

In [0]:

dbutils.widgets.text("start_date", "2026-01-01", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2026-08-15", "End Date (YYYY-MM-DD)")
dbutils.widgets.text("latitude", "-12.0432", "Latitude")
dbutils.widgets.text("longitude", "-77.0282", "Longitude")
dbutils.widgets.text("hourly_vars", "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code", "Hourly Variables")
dbutils.widgets.text("output_dir", "/Volumes/proyecto_final_prueba/raw/weather", "Output Directory")
dbutils.widgets.text("api_endpoint", "https://archive-api.open-meteo.com/v1/archive", "API endpoint")
dbutils.widgets.text("timeout", "30", "Time Out")

In [0]:
start_date = pendulum.parse(dbutils.widgets.get("start_date")).date()
end_date = pendulum.parse(dbutils.widgets.get("end_date")).date()
latitude = dbutils.widgets.get("latitude")
longitude = dbutils.widgets.get("longitude")
hourly_vars = dbutils.widgets.get("hourly_vars")
output_dir = Path(dbutils.widgets.get("output_dir"))
api_endpoint = dbutils.widgets.get("api_endpoint")
timeout = int(dbutils.widgets.get("timeout"))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6785238866360871>, line 1
----> 1 start_date = pendulum.parse(dbutils.widgets.get("start_date")).date()
      2 end_date = pendulum.parse(dbutils.widgets.get("end_date")).date()
      3 latitude = dbutils.widgets.get("latitude")

NameError: name 'pendulum' is not defined

In [0]:
save_files =[]

current_date = start_date
session = requests.Session()

while current_date <= end_date:
    date_str = current_date.to_date_string()

    daily_output_dir = (output_dir / f'{current_date.year}/{current_date.month}/{current_date.day}')
    daily_output_dir.mkdir(parents=True, exist_ok=True)

    file_path = daily_output_dir / f'weather_{date_str}.json'

    api_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "start_date": date_str,
        "end_date": date_str
    }

    try:
        response = session.get(api_endpoint, params=api_params, timeout=timeout)
        response.raise_for_status()
    except requests.RequestException:
        current_date = current_date.add(days=1)
        continue

    file_path.write_bytes(response.content)
    save_files.append(file_path)
    current_date = current_date.add(days=1)

print(f"Archivos guardados: {len(save_files)}")

Archivos guardados: 227
